In [1]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchinfo import summary

from utils.logger import Logger

In [2]:
pd.set_option('display.max_columns', None)

In [ ]:
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')
orthogonal_df = pd.read_csv('../data/raw/orthogonal.csv')

In [4]:
train_df.columns = list(map(lambda x: x.replace(" ", "_").lower(), train_df.columns))
test_df.columns = list(map(lambda x: x.replace(" ", "_").lower(), test_df.columns))
orthogonal_df.columns = list(map(lambda x: x.replace(" ", "_").lower(), orthogonal_df.columns))

In [5]:
deleted_cols = [col for col in test_df.columns if col not in train_df.columns]
print(f"{deleted_cols = }")

deleted_cols = ['clinpred_score', 'revel_score', 'varity_r', 'metasvm_score', 'metalr_score', 'vest4_score', 'm-cap_score', 'mutpred_score', 'primateai_score', 'mutationassessor_score', 'list-s2_score', 'sift4g_score', 'dann_rankscore', 'mutationtaster_score']


In [6]:
test_df_filtered = test_df.drop(columns=deleted_cols)
orthogonal_df_filtered = orthogonal_df.drop(columns=deleted_cols)

In [7]:
print(f"{train_df.shape = }")
print(f"{test_df_filtered.shape = }")
print(f"{orthogonal_df_filtered.shape = }")

train_df.shape = (77002, 77)
test_df_filtered.shape = (8666, 77)
orthogonal_df_filtered.shape = (13383, 77)


In [8]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoder.fit(train_df['chr'])
train_df['chr'] = encoder.transform(train_df['chr'])
test_df_filtered['chr'] = encoder.transform(test_df_filtered['chr'])
orthogonal_df_filtered['chr'] = encoder.transform(orthogonal_df_filtered['chr'])

In [9]:
train_df['class'] = train_df['class'].replace(-1, 0)
test_df_filtered['class'] = test_df_filtered['class'].replace(-1, 0)
orthogonal_df_filtered['class'] = orthogonal_df_filtered['class'].replace(-1, 0)

In [ ]:
from w2v_model import KmerGenerator

ref_model_path = "../models/global_models/ref_word2vec.model"
alt_model_path = "../models/global_models/alt_word2vec.model"

ref_model = Word2Vec.load(ref_model_path)
alt_model = Word2Vec.load(alt_model_path)


kmer_gen = KmerGenerator(k=3)

def embed_sequence(seq, model, k=3):
    tokens = kmer_gen.generate_kmers(seq)
    vectors = [model.wv[kmer] for kmer in tokens if kmer in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

def embed_ref_alt(df):
    ref_embeddings = df['ref'].apply(lambda x: embed_sequence(x, ref_model))
    alt_embeddings = df['alt'].apply(lambda x: embed_sequence(x, alt_model))

    ref_emb_df = pd.DataFrame(ref_embeddings.tolist(), columns=[f'ref_emb_{i}' for i in range(ref_model.vector_size)], dtype=np.float32)
    alt_emb_df = pd.DataFrame(alt_embeddings.tolist(), columns=[f'alt_emb_{i}' for i in range(alt_model.vector_size)], dtype=np.float32)

    final_df = pd.concat([df.drop(columns=['ref', 'alt', 'gene']), ref_emb_df, alt_emb_df], axis=1)

    return final_df

In [11]:
train_df = embed_ref_alt(train_df)
test_df_filtered = embed_ref_alt(test_df_filtered)
orthogonal_df_filtered = embed_ref_alt(orthogonal_df_filtered)

In [12]:
X_train = train_df.drop(['class'], axis=1)
y_train = train_df['class']

X_test = test_df_filtered.drop(['class'], axis=1)
y_test = test_df_filtered['class']

X_orthogonal = orthogonal_df_filtered.drop(['class'], axis=1)
y_orthogonal = orthogonal_df_filtered['class']

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_orthogonal = scaler.transform(X_orthogonal)

In [14]:
class ClinVarDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [15]:
train_ds = ClinVarDataset(X_train, y_train)
test_ds = ClinVarDataset(X_test, y_test)
orthogonal_ds = ClinVarDataset(X_orthogonal, y_orthogonal)

In [16]:
BATCH_SIZE = 64

train_ds_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_ds_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
orthogonal_ds_loader = DataLoader(orthogonal_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

In [17]:
class PathogenicityClassifier(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
        )
        self.output = nn.Linear(16, output_size)

    def forward(self, x):
        x = self.model(x)
        return self.output(x)

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
EPOCH = 100

model = PathogenicityClassifier(input_size=X_train.shape[1], output_size=2)
model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6, threshold=1e-3)

In [ ]:
best_model_checkpoint = {'loss': float('inf'), 'epoch': 0, 'path': '../temp/best_model.pt'}

for epoch in range(EPOCH):
    epoch_loss_train = 0.0
    correct_train = 0
    total_train = 0

    # Training Phase
    model.train()
    for batch_features, batch_labels in train_ds_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        output = model(batch_features)
        loss = criterion(output, batch_labels)
        
        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        epoch_loss_train += loss.item()

        _, predicted = torch.max(output, 1)
        correct_train += (predicted == batch_labels).sum().item()
        total_train += batch_labels.size(0)
    
    avg_loss_train = epoch_loss_train / len(train_ds_loader)
    accuracy_train = correct_train / total_train

    # Evaluation Phase
    epoch_loss_test = 0.0
    correct_test = 0
    total_test = 0

    model.eval()
    with torch.no_grad():
        for batch_features, batch_labels in test_ds_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            output = model(batch_features)
            loss = criterion(output, batch_labels)
            epoch_loss_test += loss.item()

            _, predicted = torch.max(output, 1)
            correct_test += (predicted == batch_labels).sum().item()
            total_test += batch_labels.size(0)
    
    avg_loss_test = epoch_loss_test / len(test_ds_loader)
    accuracy_test = correct_test / total_test

    print(f"Epoch [{epoch+1}/{EPOCH}]: Training Accuracy: {accuracy_train:.4f}, Loss: {avg_loss_train:.4f}; Test Accuracy: {accuracy_test:.4f}, Loss: {avg_loss_test:.4f}; learning_rate: {optimizer.param_groups[0]['lr']:.6f}")

    # Checkpoint
    if avg_loss_test < best_model_checkpoint['loss']:
        best_model_checkpoint['loss'] = avg_loss_test
        best_model_checkpoint['epoch'] = epoch
        torch.save(model.state_dict(), best_model_checkpoint['path'])

    scheduler.step(avg_loss_test)


# Load best model
model.load_state_dict(torch.load(best_model_checkpoint['path']))
model.eval()
print(f"\n\nRestored best model weights from epoch {best_model_checkpoint['epoch']} with loss={best_model_checkpoint['loss']}.")

In [ ]:
# logger = Logger()

In [ ]:
# logger.log_info(summary(model, input_size=(BATCH_SIZE, ), device=device))

In [ ]:
# summary(model, device=device)

KeyboardInterrupt: 

In [ ]:
def evaluate(model, ds_loader, device=None):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_labels in ds_loader:
            if device:
                batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)

            total += batch_labels.shape[0]
            correct += (predicted == batch_labels).sum().item()

        accuracy = correct/total
    return accuracy

In [ ]:
train_acc = evaluate(model, train_ds_loader, device)
test_acc = evaluate(model, test_ds_loader, device)
valid_acc = evaluate(model, orthogonal_ds_loader, device)

print(f"{train_acc = }")
print(f"{test_acc = }")
print(f"{valid_acc = }")

In [ ]:
# train_acc = 0.9581958910158178
# test_acc = 0.9552273251788599
# valid_acc = 0.9432115370245834

In [ ]:
# # test evaluation code
# total = 0
# correct = 0

# with torch.no_grad():

#   for batch_features, batch_labels in test_ds_loader:

#     # move data to gpu
#     batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

#     outputs = model(batch_features)

#     _, predicted = torch.max(outputs, 1)

#     total = total + batch_labels.shape[0]

#     correct = correct + (predicted == batch_labels).sum().item()

# print(correct/total)

In [ ]:
torch.save(model, "../models/global_models/baseline_model.pt")